# India Tech Job Market Analysis — Step 2: Data Inspection & Quality Check

**Project stack:** Python + SQL + Power BI

This notebook inspects the job-market dataset before analysis.

**Important:** keep the original CSV unchanged. Create a cleaned copy only after these checks.

In [ ]:
import pandas as pd
import numpy as np

# Keep the CSV in the same folder as this notebook, or change the path below.
FILE_PATH = "india_job_market_2024_2026.csv"

df = pd.read_csv(FILE_PATH)
df.head()

In [ ]:
# 1. Basic structure
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
"# 2. Data types
df.info()

In [ ]:
# 3. Missing values
missing = df.isna().sum().sort_values(ascending=False)
print(missing)
print("\nTotal missing cells:", missing.sum())

In [ ]:
# 4. Duplicate checks
print("Exact duplicate rows:", df.duplicated().sum())

if "Job_ID" in df.columns:
    print("Duplicate Job_IDs:", df["Job_ID"].duplicated().sum())

In [ ]:
# 5. Unique values in important categorical columns
categorical_cols = [
    "Job_Title", "Company", "Company_Type", "Industry", "City",
    "Location_Tier", "Experience_Level", "Job_Type",
    "Work_Mode", "Education_Required"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique:", df[col].nunique())
    print(df[col].value_counts().head(10))

In [ ]:
# 6. Numeric summary
numeric_cols = ["Salary_LPA", "Openings", "Applicants", "Company_Rating"]
df[numeric_cols].describe().T

In [ ]:
# 7. Date validation
df["Date_Posted"] = pd.to_datetime(df["Date_Posted"], errors="coerce")

print("Invalid dates:", df["Date_Posted"].isna().sum())
print("Earliest date:", df["Date_Posted"].min())
print("Latest date:", df["Date_Posted"].max())

In [ ]:
# 8. Business-rule checks
print("Openings <= 0:", (df["Openings"] <= 0).sum())
print("Applicants <= 0:", (df["Applicants"] <= 0).sum())
print("Applicants < Openings:", (df["Applicants"] < df["Openings"]).sum())
print("Salary <= 0:", (df["Salary_LPA"] <= 0).sum())

# Inspect unusually low salaries rather than deleting them automatically.
print("\nRows with Salary_LPA <= 1:")
display(
    df.loc[df["Salary_LPA"] <= 1,
           ["Job_Title", "Experience_Level", "City", "Salary_LPA", "Openings", "Applicants"]]
    .head(20)
)

In [ ]:
# 9. Create useful derived fields for the analysis
df["Applicants_per_Opening"] = df["Applicants"] / df["Openings"]
df["Year"] = df["Date_Posted"].dt.year
df["Month"] = df["Date_Posted"].dt.month
df["Year_Month"] = df["Date_Posted"].dt.to_period("M").astype(str)
df["Skill_Count"] = df["Skills_Required"].str.split(",").str.len()

df[["Applicants_per_Opening", "Year", "Month", "Year_Month", "Skill_Count"]].head()

In [ ]:
# 10. Quick project-level summary
summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "unique_jobs": df["Job_ID"].nunique(),
    "unique_companies": df["Company"].nunique(),
    "unique_job_titles": df["Job_Title"].nunique(),
    "avg_salary_lpa": df["Salary_LPA"].mean(),
    "median_salary_lpa": df["Salary_LPA"].median(),
    "avg_applicants_per_opening": df["Applicants_per_Opening"].mean(),
}
summary

## Initial interpretation

Do **not** delete the low-salary rows automatically. The inspection may reveal unusual records that need a documented treatment rule.

Also note that this dataset appears to be structured for analysis rather than a clean mirror of live job-board data. For the portfolio project, we should describe it accurately as a **public job-market dataset** and avoid claiming that its figures represent the entire Indian job market.